# Fine-tune Laya on MELD (Kaggle, GPU T4 ×2)

Fine-tunes [Laya](https://huggingface.co/convaiinnovations/laya) (421M) on the MELD training split
(9,989 *Friends* utterances → 19,978 emotion + sentiment decisions), then scores the full MELD test
set with the same questions and metrics as the repo's baselines.

Adapted from the upstream [Laya fine-tuning notebook](https://github.com/NandhaKishorM/laya/blob/9d955671415fc19f069b9cc998928075c1f255ec/notebooks/laya_finetune_typed_decisions_2xT4_kaggle.ipynb)
(RLCD policy gradient + soft cross-entropy, DDP on two T4s). Changes from upstream:

- **Data:** MELD records built by `laya_meld.finetune`, so training inputs are token-for-token what inference builds.
- **Model selection:** the MELD **dev** split is scored after every epoch, and the epoch with the best dev emotion weighted F1 is kept. The test split is never used for any choice.
- **Calibration:** temperatures are fitted on dev instead of a slice carved from train.

**Notebook settings:** Accelerator **GPU T4 x2**, Internet **On**. Expected run time: roughly 20–40 minutes.

**Outputs** (in `/kaggle/working`):
- `laya-meld-v1/`: the fine-tuned checkpoint (loads with `laya.Agent(path)`).
- `results/laya_finetuned_v1.json` and `results/predictions/laya_finetuned_v1.jsonl`: drop these into the repo's `results/`.

## 1. Check both GPUs

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
n_gpu = torch.cuda.device_count()
assert n_gpu >= 2, f"Expected 2 GPUs, found {n_gpu}. Set Accelerator to GPU T4 x2."
print(f"{n_gpu} GPUs ready")

## 2. Install Laya and get the repo code

In [ ]:
!pip install -q "laya==0.3.21"
REPO = "https://github.com/ManasviAtGitHub/ai-agents-and-laya-system-meld"
!rm -rf /kaggle/working/repo && git clone -q --depth 1 {REPO} /kaggle/working/repo
REPO_COMMIT = !git -C /kaggle/working/repo rev-parse HEAD
REPO_COMMIT = REPO_COMMIT[0]

import sys
sys.path.insert(0, "/kaggle/working/repo/src")   # the package itself; its CPU-only deps aren't needed here
import laya
print("laya", laya.__version__, "| repo commit", REPO_COMMIT)

## 3. Build MELD training items

In [ ]:
import os, json, torch
from huggingface_hub import snapshot_download
from transformers import AutoTokenizer
from laya.agent import _fix_tokenizer_config
from laya_meld import data, finetune

list(data.download())

MODEL_ID = "convaiinnovations/laya"
model_dir = snapshot_download(MODEL_ID)
_fix_tokenizer_config(model_dir)
tok = AutoTokenizer.from_pretrained(os.path.join(model_dir, "tokenizer"))

for split in ("train", "dev"):
    items, skipped = finetune.tokenize(finetune.records(split), tok)
    torch.save(items, f"/kaggle/working/{split}_items.pt")
    print(f"{split}: {len(items)} items, {skipped} skipped")

## 4. Training script (DDP, RLCD)

In [ ]:
%%writefile /kaggle/working/train_ddp.py
import os, sys, time, json, random
import numpy as np
import torch
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP
from safetensors.torch import load_file, save_file
from transformers import AutoTokenizer
from laya.common import build_model, proper_reward

sys.path.insert(0, "/kaggle/working/repo/src")
from laya_meld import finetune

EPOCHS = 3
MICRO_BATCH = 8       # sequences per forward pass per GPU
GRAD_ACCUM = 4        # effective batch 8 x 2 GPUs x 4 = 64
GROUP_SIZE = 4        # GRPO baseline samples
LR_ENCODER = 2.5e-5
LR_HEAD = 1.0e-4
SIGMA_START, SIGMA_END = 0.4, 0.1


def collate(items, pad_id):
    n, L = len(items), max(len(it["ids"]) for it in items)
    kmax = max(len(it["markers"]) for it in items)
    ids = torch.full((n, L), pad_id, dtype=torch.long)
    att = torch.zeros((n, L), dtype=torch.long)
    mpos = torch.zeros((n, kmax), dtype=torch.long)
    mmask = torch.zeros((n, kmax), dtype=torch.bool)
    target = torch.zeros((n, kmax), dtype=torch.float32)
    for i, it in enumerate(items):
        ids[i, :len(it["ids"])] = torch.tensor(it["ids"])
        att[i, :len(it["ids"])] = 1
        k = len(it["markers"])
        mpos[i, :k] = torch.tensor(it["markers"])
        mmask[i, :k] = True
        target[i, :len(it["target"])] = torch.tensor(it["target"], dtype=torch.float32)
    return {"input_ids": ids, "attention_mask": att, "marker_pos": mpos, "marker_mask": mmask,
            "target": target, "qtype": torch.tensor([it["qtype"] for it in items])}


def forward(model, b, device):
    with torch.autocast("cuda", dtype=torch.float16):
        return model(b["input_ids"].to(device), b["attention_mask"].to(device), b["marker_pos"].to(device),
                     b["marker_mask"].to(device), b["qtype"].to(device))


def dev_logits(model, items, pad_id, device, bs=64):
    model.eval()
    out = []
    with torch.no_grad():
        for i in range(0, len(items), bs):
            chunk = items[i:i + bs]
            logits, _ = forward(model, collate(chunk, pad_id), device)
            logits = logits.float().cpu().numpy()
            out += [logits[r, :len(it["markers"])] for r, it in enumerate(chunk)]
    model.train()
    return out


def weighted_f1(gold, pred):
    total = 0.0
    for c in set(gold):
        tp = sum(g == p == c for g, p in zip(gold, pred))
        fp = sum(p == c != g for g, p in zip(gold, pred))
        fn = sum(g == c != p for g, p in zip(gold, pred))
        total += (2 * tp / (2 * tp + fp + fn) if tp else 0.0) * gold.count(c)
    return total / len(gold)


def dev_scores(logits, items):
    scores = {}
    for qid in ("emotion", "sentiment"):
        pairs = [(it["label"], int(np.argmax(z))) for z, it in zip(logits, items) if it["qid"] == qid]
        scores[qid] = weighted_f1([g for g, _ in pairs], [p for _, p in pairs])
    return scores


def fit_temperature(logits, items):
    """One temperature for choice questions, fitted on dev by minimising NLL."""
    kmax = max(len(z) for z in logits)
    Z = torch.full((len(logits), kmax), -1e4)
    T = torch.zeros((len(logits), kmax))
    for i, (z, it) in enumerate(zip(logits, items)):
        Z[i, :len(z)] = torch.tensor(z)
        T[i, :len(it["target"])] = torch.tensor(it["target"])
    log_t = torch.zeros(1, requires_grad=True)
    opt = torch.optim.LBFGS([log_t], lr=0.1, max_iter=100)

    def closure():
        opt.zero_grad()
        loss = -(T * torch.log_softmax(Z / log_t.exp(), -1)).sum(-1).mean()
        loss.backward()
        return loss

    opt.step(closure)
    return float(torch.clamp(log_t.exp(), 0.5, 5.0).item())   # laya clamps to [0.5, 5] at load


def save(model, tok, cfg, path, extra=None):
    os.makedirs(path, exist_ok=True)
    save_file({k: v.half().contiguous().cpu() for k, v in model.state_dict().items()},
              os.path.join(path, "model.safetensors"))
    model.encoder.config.save_pretrained(os.path.join(path, "encoder"))
    tok.save_pretrained(os.path.join(path, "tokenizer"))
    with open(os.path.join(path, "rl_agent_config.json"), "w") as f:
        json.dump({**cfg, **(extra or {})}, f, indent=2)


def main():
    dist.init_process_group("nccl")
    rank, world = dist.get_rank(), dist.get_world_size()
    local_rank = int(os.environ.get("LOCAL_RANK", "0"))
    torch.cuda.set_device(local_rank)
    device = torch.device("cuda", local_rank)
    model_dir, output_dir = sys.argv[1], sys.argv[2]

    with open(os.path.join(model_dir, "rl_agent_config.json")) as f:
        cfg = json.load(f)
    cfg["max_len"], cfg["head_max_len"] = finetune.MAX_LEN, finetune.HEAD_MAX_LEN

    tok = AutoTokenizer.from_pretrained(os.path.join(model_dir, "tokenizer"))
    model = build_model(cfg, encoder_dir=os.path.join(model_dir, "encoder"))
    model.load_state_dict(load_file(os.path.join(model_dir, "model.safetensors")), strict=True)
    model.encoder.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    model.head_checkpointing = True
    model.to(device).train()
    ddp_model = DDP(model, device_ids=[local_rank], find_unused_parameters=True)

    train_items = torch.load("/kaggle/working/train_items.pt", weights_only=False)
    dev_items = torch.load("/kaggle/working/dev_items.pt", weights_only=False)
    my_items = train_items[rank::world]

    enc = [p for n, p in ddp_model.named_parameters() if "encoder." in n]
    head = [p for n, p in ddp_model.named_parameters() if "encoder." not in n]
    optimizer = torch.optim.AdamW([{"params": enc, "lr": LR_ENCODER}, {"params": head, "lr": LR_HEAD}],
                                  weight_decay=0.01)
    total_updates = (len(my_items) // (MICRO_BATCH * GRAD_ACCUM)) * EPOCHS
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(1, total_updates), eta_min=1e-6)
    scaler = torch.amp.GradScaler("cuda")

    if rank == 0:
        print(f"{len(train_items)} train items, {len(my_items)} per rank, {len(dev_items)} dev items, {EPOCHS} epochs")
    best = {"epoch": 0, "dev": {"emotion": -1.0}}
    history = []
    t0 = time.time()

    for epoch in range(EPOCHS):
        random.seed(42 + epoch + rank)
        random.shuffle(my_items)
        sigma = SIGMA_START + (SIGMA_END - SIGMA_START) * epoch / max(1, EPOCHS - 1)
        epoch_loss, n_batches = 0.0, 0
        optimizer.zero_grad(set_to_none=True)

        for b_idx in range(0, len(my_items), MICRO_BATCH):
            b = collate(my_items[b_idx:b_idx + MICRO_BATCH], tok.pad_token_id)
            logits, act = forward(ddp_model, b, device)
            logits = logits.float()
            mask = b["marker_mask"].to(device)
            k = mask.sum(-1, keepdim=True).float()
            target = b["target"].to(device)
            qtype = b["qtype"].to(device)

            # Sample noisy logits, reward them with proper scoring rules, and push the policy
            # toward the better samples; plus soft cross-entropy on the gold distribution.
            eps = torch.randn((GROUP_SIZE,) + logits.shape, device=device) * sigma * mask
            eps = (eps - eps.sum(-1, keepdim=True) / k) * mask
            z = logits.detach().unsqueeze(0) + eps
            q = torch.softmax(z.masked_fill(~mask, -1e4), -1)
            with torch.no_grad():
                r = proper_reward(q, target.unsqueeze(0), qtype, mask, w_sph=0.75, w_rps=1.0)
                adv = (r - r.mean(0, keepdim=True)) / ((r - r.mean(0, keepdim=True)).std() + 1e-6)
            logp = -(((z - logits.unsqueeze(0)) ** 2) * mask).sum(-1) / (2 * sigma ** 2)
            loss_rl = -(adv * logp).mean()
            loss_ce = -(target * torch.log_softmax(logits.masked_fill(~mask, -1e4), -1)).sum(-1).mean()
            loss = (loss_rl + loss_ce) / GRAD_ACCUM + 0.0 * act.sum()

            scaler.scale(loss).backward()
            n_batches += 1
            if n_batches % GRAD_ACCUM == 0 or b_idx + MICRO_BATCH >= len(my_items):
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(ddp_model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
                scheduler.step()
                optimizer.zero_grad(set_to_none=True)
            epoch_loss += loss.item() * GRAD_ACCUM
            if rank == 0 and n_batches % 200 == 0:
                print(f"  epoch {epoch + 1} step {n_batches} loss {loss.item() * GRAD_ACCUM:.4f} "
                      f"lr {scheduler.get_last_lr()[0]:.2e} [{time.time() - t0:.0f}s]", flush=True)

        dist.barrier()
        if rank == 0:
            scores = dev_scores(dev_logits(model, dev_items, tok.pad_token_id, device), dev_items)
            history.append({"epoch": epoch + 1, "train_loss": epoch_loss / max(1, n_batches), "dev_weighted_f1": scores})
            print(f"=== epoch {epoch + 1}: loss {epoch_loss / max(1, n_batches):.4f} | dev weighted F1 "
                  f"emotion {scores['emotion']:.3f} sentiment {scores['sentiment']:.3f} [{time.time() - t0:.0f}s]",
                  flush=True)
            if scores["emotion"] > best["dev"]["emotion"]:
                best = {"epoch": epoch + 1, "dev": scores}
                save(model, tok, cfg, os.path.join(output_dir, "best"))
                print(f"  new best (epoch {epoch + 1}) saved", flush=True)
        dist.barrier()

    if rank == 0:
        del optimizer, scaler, scheduler
        torch.cuda.empty_cache()
        model.load_state_dict(load_file(os.path.join(output_dir, "best", "model.safetensors")))
        model.to(device)
        logits = dev_logits(model, dev_items, tok.pad_token_id, device)
        temperature = fit_temperature(logits, dev_items)
        print(f"best epoch {best['epoch']} | choice temperature fitted on dev: {temperature:.3f}")
        save(model, tok, cfg, output_dir, extra={
            "fine_tuned": True, "model_name": "laya-meld-v1",
            "temperature": [temperature, 1.0, 1.0], "temperature_by_options": {},
        })
        with open(os.path.join(output_dir, "training_log.json"), "w") as f:
            json.dump({"best": best, "history": history, "epochs": EPOCHS, "micro_batch": MICRO_BATCH,
                       "grad_accum": GRAD_ACCUM, "world_size": world, "lr_encoder": LR_ENCODER,
                       "lr_head": LR_HEAD, "choice_temperature": temperature}, f, indent=2)
        print(f"saved {output_dir} in {time.time() - t0:.0f}s")
    dist.destroy_process_group()


if __name__ == "__main__":
    main()

## 5. Train on both GPUs

In [ ]:
OUTPUT_DIR = "/kaggle/working/laya-meld-v1"
!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True torchrun --standalone --nproc_per_node=2 /kaggle/working/train_ddp.py {model_dir} {OUTPUT_DIR}

## 6. Score the MELD test set
Same questions, same inputs and same metrics as `scripts/run_baseline.py` in the repo.

In [ ]:
import time, json, pathlib
from importlib.metadata import version
from laya_meld import evaluation
from laya_meld.questions import MELD_V1, state_for

agent = laya.Agent(OUTPUT_DIR, device="cuda")
test = data.load("test")

t0 = time.time()
predictions = {}
BATCH = 64
for i in range(0, len(test), BATCH):
    chunk = test[i:i + BATCH]
    outs = agent.predict_batch([state_for(u) for u in chunk], MELD_V1, batch_size=BATCH)
    for u, out in zip(chunk, outs):
        predictions[u["id"]] = evaluation.laya_prediction(u["id"], out["answers"])
print(f"scored {len(test)} utterances in {time.time() - t0:.0f}s")

results = evaluation.score(test, predictions)
print(evaluation.summary(results))

training = json.load(open(f"{OUTPUT_DIR}/training_log.json"))
out_dir = pathlib.Path("/kaggle/working/results")
evaluation.write_result("laya_finetuned_v1", "laya_finetuned", {
    "questions": "MELD_V1 (identical to ZERO_SHOT_V1)", "input": "previous line + speaker + line",
    "laya_version": version("laya"), "repo_commit": REPO_COMMIT, "trained_on": "MELD train",
    "selected_on": "MELD dev emotion weighted F1", "best_epoch": training["best"]["epoch"],
    "dev_weighted_f1": training["best"]["dev"], "choice_temperature": training["choice_temperature"],
    "hardware": "Kaggle 2x T4 (DDP) for training, 1x T4 for scoring",
}, results, results_dir=out_dir)
(out_dir / "predictions").mkdir(exist_ok=True)
with open(out_dir / "predictions" / "laya_finetuned_v1.jsonl", "w") as f:
    for u in test:
        f.write(json.dumps(predictions[u["id"]]) + "\n")
print("wrote", sorted(str(p) for p in out_dir.rglob("*.json*")))

## 7. Package outputs for download
Download `laya-meld-v1.zip` (checkpoint + training log) and the `results/` folder from the Output panel.

In [ ]:
!rm -rf {OUTPUT_DIR}/best
!cd /kaggle/working && zip -qr laya-meld-v1.zip laya-meld-v1 && ls -lh laya-meld-v1.zip
!ls -R /kaggle/working/results